In [5]:
import os
import sys
from pathlib import Path

sys.path.append(os.path.abspath(".."))

import pandas as pd

from simulation import simulate_many_from_snapshot

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"

PROCESSED_PATH = DATA_DIR / "processed"
LIVE_PATH = DATA_DIR / "live"

### live_results.csv

raw input, one row per contestant per episode

- contestant
- episode
- episode_score

✅ Done

In [4]:
live_results = pd.read_csv(LIVE_PATH / "live_results.csv")

live_results

,contestant,episode,episode_score
0,Amy Gledhill,1,18
1,Armando Iannucci,1,16
2,Joanna Page,1,12
3,Joel Dommett,1,9
4,Kumail Nanjiani,1,13


### latest_snapshot.csv

One row per contestant:

- contestant
- latest_episode (later)
- latest_episode_score
- cumulative_score
- current_rank
- predicted_final_score (later)
- win_probability
- archetype (later)
- archetype_description (later)

In [9]:
# ----- Build live_snapshot (temporary) -----

live_snapshot = (
    live_results
    .sort_values(["contestant", "episode"])
    .groupby("contestant")["episode_score"]
    .agg(["sum", "mean", "std", "count"])
    .reset_index()
    .rename(columns={
        "sum": "cumulative_score",
        "mean": "mean_score_so_far",
        "std": "std_score_so_far",
        "count": "episodes_played"
    })
)

total_episodes = 10

live_snapshot["remaining_episodes"] = (
    total_episodes - live_snapshot["episodes_played"]
)

# fix std
global_std = live_results["episode_score"].std()

live_snapshot["std_score_so_far"] = live_snapshot["std_score_so_far"].fillna(global_std)

live_snapshot.loc[
    live_snapshot["std_score_so_far"] == 0,
    "std_score_so_far"
] = global_std


# ----- Run simulations -----

live_simulations = simulate_many_from_snapshot(
    live_snapshot,
    n_simulations=1000
)


# ----- Convert to win probabilities -----

win_probs = (
    live_simulations[live_simulations["rank"] == 1]
    .groupby("contestant")
    .size()
    .reset_index(name="wins")
)

win_probs["win_probability"] = win_probs["wins"] / 1000


# ----- Build latest_snapshot -----

# cumulative score
cum_scores = (
    live_results
    .groupby("contestant")["episode_score"]
    .sum()
    .reset_index(name="cumulative_score")
)

# latest episode score
latest_ep = live_results["episode"].max()

latest_scores = (
    live_results[live_results["episode"] == latest_ep]
    [["contestant", "episode_score"]]
    .rename(columns={"episode_score": "latest_episode_score"})
)

# merge
latest_snapshot = cum_scores.merge(win_probs, on="contestant", how="left")
latest_snapshot = latest_snapshot.merge(latest_scores, on="contestant", how="left")

# rank
latest_snapshot["current_rank"] = (
    latest_snapshot["cumulative_score"]
    .rank(ascending=False, method="min")
)

latest_snapshot = latest_snapshot.fillna(0)

# sort nicely
latest_snapshot = latest_snapshot.sort_values(
    "win_probability", ascending=False
).reset_index(drop=True)


# ----- Select only the required columns -----

latest_snapshot = latest_snapshot[
    [
        "contestant",
        "latest_episode_score",
        "cumulative_score",
        "current_rank",
        "win_probability"
    ]
]

# clean types
latest_snapshot["current_rank"] = latest_snapshot["current_rank"].astype(int)


# ----- Save snapshot -----

latest_snapshot.to_csv(PROCESSED_PATH / "latest_snapshot.csv", index=False)

### score_trajectory.csv

One row per contestant per episode:

- contestant
- episode
- episode_score
- cumulative_score

### win_probability_trajectory.csv

One row per contestant per episode:

- contestant
- episode
- win_probability

## archetype_examples.csv

- archetype
- archetype_description
- example_contestant